In [ ]:
from sagemaker.workflow.function_step import step
from sagemaker.workflow.pipeline import Pipeline
import sagemaker
from sagemaker.workflow.parameters import ParameterInteger
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep
import boto3

In [ ]:
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

region = boto3.Session().region_name
print(f"Region: {region}")

TRACKING_SERVER = "mlflow-tracking-server-grupo5"
default_bucket = "s3-mlflow-artifacts-mlflow-tracking-server-grupo5-01"
default_prefix = f"sagemaker/clients-attrition/{user}"
default_path = default_bucket + "/" + default_prefix
athena_database = "glue-catalog-database-utec-bank-01"

user = "grupo5"
role = sagemaker.get_execution_role()


In [ ]:
sagemaker_session = sagemaker.Session(default_bucket=default_bucket,
                                      default_bucket_prefix=default_prefix)
# #Pipeline configuration
instance_type = "ml.m5.large"
pipeline_name = f"pipeline-train-{user}"
model_name = f"credit-card-fraud-detection-{user}"

#MLFlow configuration
tracking_server_arn = f"arn:aws:sagemaker:${region}:${account_id}:mlflow-tracking-server/${TRACKING_SERVER}"
experiment_name = f"pipeline-train-exp-{user}"

In [ ]:
cod_month_start = ParameterInteger(name="PeriodoCargaInicio")
cod_month_end = ParameterInteger(name="PeriodoCargaFin")

In [ ]:
@step(
    name="DataPull",
    instance_type=instance_type,
    dependencies="./data_pull_requirements.txt"
)
def data_pull(experiment_name: str, run_name: str,
              cod_month_start: int, cod_month_end: int) -> tuple[str, str, str]:
    import awswrangler as wr
    import mlflow

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)
    TARGET_COL = "attrition"
    query = """
        SELECT  id_correlativo
                ,flg_bancarizado
                ,edad
                ,antiguedad
                ,attrition
                ,flg_nomina
                ,cod_mes
                ,sdo_activo_menos0
                ,sdo_activo_menos1
                ,sdo_activo_menos2
                ,sdo_activo_menos3
        FROM    "glue-catalog-database-utec-bank-01"."data_origen"
        WHERE   cod_mes between {} and {}
    """.format(cod_month_start, cod_month_end)
    train_s3_path = f"s3://{default_path}/data_train/trained_clientes_pipeline.csv"
    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        with mlflow.start_run(run_name="DataPull", nested=True):
            df = wr.athena.read_sql_query(sql=query, database=athena_database)
            df.to_csv(train_s3_path, index=False)
            mlflow.log_input(
                mlflow.data.from_pandas(df, train_s3_path,
                                        targets=TARGET_COL),
                context="DataPull"
            )
    return train_s3_path, experiment_name, run_id

In [ ]:
data_pull_step = data_pull(experiment_name=experiment_name,
                           run_name=ExecutionVariables.PIPELINE_EXECUTION_ID,
                           cod_month_start=cod_month_start,
                           cod_month_end=cod_month_end)

# agregar mas steps

In [ ]:
steps = [
  data_pull_step,
  # model_training_step,
  # conditional_register_step
]
pipeline = Pipeline(name=pipeline_name,
                    steps=steps,
                    parameters=[cod_month_start, cod_month_end])
pipeline.upsert(role_arn=role)

In [ ]:
pipeline.start(parameters={"PeriodoCargaInicio": 202411,
                           "PeriodoCargaFin": 202412},
               execution_display_name="test-training-full-2",
               execution_description="Testando training full 2")